# 코스피·코스닥 전체 상장사 시점고정(PIT) 백테스트 DB 빌더 (Google Colab)

기존 374개사 표본의 **사후 종목 선별 편향**을 없애기 위해, **코스피·코스닥 전체 보통주(시총 하한 0원, ~2,300개사)**의 최근 12년(`FY2014~FY2025`) OpenDART 배당·재무·발행주식수·자사주 공시를 수집해 **Google Drive에 나만의 풀 유니버스 PIT DB(`full_kr_fundamentals_pit.jsonl.gz`)**를 구축하는 노트북입니다.

### 편향 최소화 설계
1. **현재 시총 하한 없음 (`MIN_MCAP_EOK = 0`)**: 2018년 당시엔 시총 3,000억 이상이었지만 이후 주가 폭락으로 소형주가 된 기업도 빠짐없이 수집합니다.
2. **과거 시점 배당 이력 기준 (미래 누출 차단)**: 2025년 현재 배당주만 고르지 않고, **`2014~2018`부터 `2021~2025`까지 어느 5년 구간에서든 4년 이상 배당을 지급했던 모든 기업**을 2차 정밀 수집합니다. *(과거엔 우량 배당주였으나 이후 배당 삭감·중단·폭락한 기업 포함!)*
3. **시점고정(PIT) 시가총액 & 액면분할 보정**: 연도별 발행주식수와 분할 비율을 함께 저장해 `colab_alpha_lab.ipynb`에서 **매수 당시 실제 시가총액 기준**으로 필터링할 수 있습니다.
4. **무료 API의 정직한 한계**: 네이버 금융은 상장폐지된 종목의 과거 일별 주가를 제공하지 않으므로, 상폐 기업은 무료 API만으로 복원할 수 없습니다. *(대신 `colab_alpha_lab.ipynb`에서 보수적 상폐 페널티 스트레스 테스트로 보정합니다.)*

### OpenDART 일일 한도(20,000건) 자동 이어받기 (약 2일 분할 수집)
- OpenDART 공식 일일 호출 정책(1일 20,000건)을 준수하며, 무배당 종목 조기 생략 및 자사주 조건부 조회 최적화 덕분에 **총 호출량이 약 18,000~19,500건**으로 줄어, **단일 키로도 대부분 하루(1회 실행) 만에 완료**됩니다. *(소량 남더라도 Drive 체크포인트로 다음 날 1분 만에 이어서 마무리)*
- 중간에 한도(`020`)에 도달하거나 화면이 꺼져 끊겨도 **Google Drive 체크포인트에 즉시 안전 저장**되므로, 다음 날 다시 **`런타임 → 모두 실행`**만 누르시면 끊긴 종목부터 이어서 완료됩니다.
- 완료되면 `MyDrive/Colab Notebooks/dividend_alpha_lab/full_kr_fundamentals_pit.jsonl.gz`가 생성되고, 이후 **`colab_alpha_lab.ipynb`를 열면 자동으로 374개사 대신 풀 유니버스 DB를 불러와 분석**합니다!


In [ ]:
# ==============================================================================
# 1. 환경 설정 및 OpenDART API 키 안전 로드 (단일 키 공식 정책 준수)
# ==============================================================================
import os, sys, re, io, gzip, json, math, time, zipfile, getpass, warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

import numpy as np
import pandas as pd
import requests

def load_dart_api_key() -> str:
    key = ""
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get("DART_API_KEY") or "").strip()
        except Exception:
            key = ""
    if not key:
        key = os.environ.get("DART_API_KEY", "").strip()
    if not key:
        key = getpass.getpass("OpenDART API Key(DART_API_KEY)를 입력하세요: ").strip()
    if not key:
        raise RuntimeError("DART_API_KEY가 필요합니다. Colab Secrets에 DART_API_KEY를 등록해 주세요.")
    return key

DART_API_KEY = load_dart_api_key()
print(f"[완료] DART_API_KEY 로드 완료 (길이 {len(DART_API_KEY)}자, 마스킹 보호) | Colab={IN_COLAB}")


In [ ]:
# ==============================================================================
# 2. 풀 유니버스 PIT 빌더 설정 & Google Drive 체크포인트 연결
# ==============================================================================
BUILDER_CONFIG = {
    "MARKETS": ["KOSPI", "KOSDAQ"],
    "MIN_MCAP_EOK": 0,              # 0 = 현재 시총 하한 없음 (사후 크기 편향 제거!)
    "LATEST_FY": 2025,              # 최신 확정 사업연도 (매년 4~5월 +1 갱신)
    "PIT_START_FY": 2018,           # PIT 스냅샷 시작 연도 (2019-04 백테스트 출발)
    "PIT_END_FY": 2025,             # PIT 스냅샷 종료 연도
    "HIST_MIN_PAID_IN_5Y": 4,       # 2014~2025 중 어느 5년 구간에서든 4년↑ 배당 지급 시 2차 수집 대상

    # 호출 안전장치 (OpenDART 공식 일일 한도 20,000건 준수)
    "MAX_CALLS_PER_RUN": 19500,     # 여유분 500건 남기고 안전 중단 -> 다음 날 이어받기
    "WORKERS": 6,

    # Google Drive 저장 경로 (colab_alpha_lab.ipynb와 동일 경로 공유!)
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_alpha_lab",
    "OUTPUT_FILENAME": "full_kr_fundamentals_pit.jsonl.gz",
    "CACHE_DIR": "./_cache_pit_builder",
    "FORCE_RESTART": False,         # True로 바꾸면 기존 체크포인트 무시하고 처음부터 재수집
}

CACHE_DIR = Path(BUILDER_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = None
if IN_COLAB and BUILDER_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(BUILDER_CONFIG["DRIVE_DIR"])
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 체크포인트·최종 DB 경로: {DRIVE_ROOT}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 임시 저장소 사용 ({type(e).__name__}: {e})")

def save_ckpt(name: str, obj: dict) -> None:
    raw = gzip.compress(json.dumps(obj, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    (CACHE_DIR / name).write_bytes(raw)
    if DRIVE_ROOT is not None:
        try:
            tmp = DRIVE_ROOT / (name + ".tmp")
            dst = DRIVE_ROOT / name
            tmp.write_bytes(raw)
            tmp.replace(dst)
        except Exception as e:
            print(f"  [주의] Drive 저장 지연({type(e).__name__}) — 로컬 보존됨")

def load_ckpt(name: str) -> dict:
    if BUILDER_CONFIG["FORCE_RESTART"]:
        return {}
    for base in ([DRIVE_ROOT, CACHE_DIR] if DRIVE_ROOT is not None else [CACHE_DIR]):
        p = base / name
        if p.exists() and p.stat().st_size > 0:
            try:
                return json.loads(gzip.decompress(p.read_bytes()).decode("utf-8"))
            except Exception:
                continue
    return {}

print(f"[설정] 대상: {BUILDER_CONFIG['MARKETS']} 전 보통주(시총 하한 {BUILDER_CONFIG['MIN_MCAP_EOK']}억) | FY{BUILDER_CONFIG['PIT_START_FY']}~FY{BUILDER_CONFIG['PIT_END_FY']}")


In [ ]:
# ==============================================================================
# 3. 1단계: 코스피·코스닥 전체 보통주 목록 + DART 고유번호 매핑
# ==============================================================================
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-pit-builder)"}
_EXCLUDE_RE = re.compile(r"(ETF|ETN|리츠|스팩|SPAC|인프라|선박투자|펀드|KODEX|TIGER|ACE|KBSTAR|SOL|ARIRANG|HANARO|PLUS|RISE)")
_PATH_HINT = [
    ("securities", r"증권"),
    ("financial_holding", r"캐피탈|카드|투자|한국금융지주|메리츠금융"),
    ("bank", r"은행|금융지주|신한지주|우리금융|하나금융|BNK|DGB|iM금융|JB금융|KB금융|금융$"),
    ("insurance", r"생명(?!과학|공학|자원)|화재|보험|손보|코리안리"),
]

def guess_path(name: str) -> str:
    for p, pat in _PATH_HINT:
        if re.search(pat, str(name)):
            return p
    return "general"

def _num(v):
    if v in (None, "", "-", "－", "N/A"):
        return None
    s = str(v).replace(",", "").replace("배", "").replace("원", "").replace("%", "").strip()
    try:
        x = float(s)
        return None if (math.isnan(x) or math.isinf(x)) else x
    except ValueError:
        return None

def _num_dash0(v):
    return 0.0 if str(v).strip() in ("-", "－") else _num(v)

def _stock_class(s: str) -> str | None:
    s = str(s or "")
    if "보통" in s or "의결권 있는" in s or "의결권있는" in s:
        return "common"
    if "우선" in s or "종류주" in s or "의결권 없는" in s or "의결권없는" in s:
        return "pref"
    return None

def fetch_naver_market(market: str) -> pd.DataFrame:
    rows, page = [], 1
    while True:
        r = requests.get(f"https://m.stock.naver.com/api/stocks/marketValue/{market}",
                         params={"page": page, "pageSize": 100}, headers=_HEADERS, timeout=20)
        r.raise_for_status()
        stocks = r.json().get("stocks") or []
        if not stocks:
            break
        for s in stocks:
            rows.append({
                "ticker": str(s["itemCode"]).zfill(6),
                "name": s["stockName"],
                "close": _num(s.get("closePrice")),
                "mcap_eok": _num(s.get("marketValue")) or 0.0,
                "end_type": s.get("stockEndType"),
                "market": market,
            })
        page += 1
        time.sleep(0.08)
    df = pd.DataFrame(rows)
    df["is_pref"] = df["ticker"].str[-1] != "0"
    df["excluded"] = (df["end_type"] != "stock") | df["name"].str.contains(_EXCLUDE_RE)
    return df

def fetch_corp_codes() -> dict[str, str]:
    ck = load_ckpt("ckpt_corp_codes.json.gz")
    if ck:
        return ck
    r = requests.get("https://opendart.fss.or.kr/api/corpCode.xml", params={"crtfc_key": DART_API_KEY}, timeout=90)
    if r.status_code != 200:
        raise RuntimeError(f"OpenDART corpCode.xml HTTP {r.status_code}")
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        root = ET.fromstring(z.read(z.namelist()[0]))
    mapping = {}
    for el in root.iter("list"):
        sc = (el.findtext("stock_code") or "").strip()
        cc = (el.findtext("corp_code") or "").strip()
        if sc and cc:
            mapping[sc.zfill(6)] = cc
    save_ckpt("ckpt_corp_codes.json.gz", mapping)
    return mapping

print("[1단계] 코스피·코스닥 전체 보통주 목록 & DART 고유번호 수집 중...")
all_mkt = pd.concat([fetch_naver_market(m) for m in BUILDER_CONFIG["MARKETS"]], ignore_index=True).drop_duplicates("ticker")
CORP_MAP = fetch_corp_codes()

UNI_DF = all_mkt[(~all_mkt["excluded"]) & (~all_mkt["is_pref"]) & (all_mkt["mcap_eok"] >= BUILDER_CONFIG["MIN_MCAP_EOK"])].copy()
UNI_DF["corp_code"] = UNI_DF["ticker"].map(CORP_MAP)
UNI_DF = UNI_DF[UNI_DF["corp_code"].notna()].reset_index(drop=True)
UNI_DF["path"] = UNI_DF["name"].map(guess_path)
print(f"[완료] 전체 상장 {len(all_mkt):,} -> 보통주 & DART 매핑 완료: {len(UNI_DF):,}종목 ({UNI_DF['market'].value_counts().to_dict()})")


In [ ]:
# ==============================================================================
# 4. 2단계: 전 종목 12년 배당 이력 고속 스캔 (~9,200회 호출, 50종목마다 Drive 저장)
# ==============================================================================
DART_BASE = "https://opendart.fss.or.kr/api"
DART_STAT = {"calls": 0, "limit_hit": False}

def dart_get(endpoint: str, corp_code: str, bsns_year: int, extra: dict | None = None) -> list[dict] | None:
    """OpenDART 호출 (일일 한도 도달 시 None 반환하고 안전 중단)."""
    if DART_STAT["limit_hit"] or DART_STAT["calls"] >= BUILDER_CONFIG["MAX_CALLS_PER_RUN"]:
        DART_STAT["limit_hit"] = True
        return None
    params = {"crtfc_key": DART_API_KEY, "corp_code": corp_code, "bsns_year": str(bsns_year), "reprt_code": "11011"}
    if extra:
        params.update(extra)
    for attempt in range(4):
        try:
            DART_STAT["calls"] += 1
            r = requests.get(f"{DART_BASE}/{endpoint}.json", params=params, timeout=30)
            if r.status_code != 200:
                time.sleep(1.2 * (attempt + 1))
                continue
            j = r.json()
            st = j.get("status")
            if st == "020":
                DART_STAT["limit_hit"] = True
                return None
            if st == "013":
                return []
            if st == "000":
                return j.get("list") or []
            return []
        except Exception:
            time.sleep(1.0 * (attempt + 1))
    return []

def parse_alot_12y(corp_code: str, last_fy: int) -> dict[int, dict] | None:
    """2단계 고속 배당 조회: 먼저 (last_fy, last_fy-6) 2회만 확인해 두 구간 모두 무배당이면 나머지 2회 생략 (호출 ~2,200회 절감, 편향 0%)."""
    by_year: dict[int, dict] = {}
    probe_years = [y for y in (last_fy, last_fy - 6) if y >= 2015]
    rest_years = [y for y in (last_fy - 3, last_fy - 9) if y >= 2015]

    def _fetch_years(year_list):
        for cy in year_list:
            rows = dart_get("alotMatter", corp_code, cy)
            if rows is None:
                return False
            if not rows:
                continue
            st_dates = sorted({str(r.get("stlm_dt") or "").strip() for r in rows if str(r.get("stlm_dt") or "").strip() not in ("", "-")})
            if len(st_dates) > 1:
                rows = [r for r in rows if str(r.get("stlm_dt") or "").strip() == st_dates[-1]]
            cfs_ni = any("당기순이익" in str(r.get("se", "")) and "연결" in str(r.get("se", "")) for r in rows)
            for col, lag in (("thstrm", 0), ("frmtrm", 1), ("lwfr", 2)):
                y = cy - lag
                if y in by_year and by_year[y].get("dps_common") is not None:
                    continue
                out = by_year.setdefault(y, {})
                period_sum, total_dash0 = 0.0, False
                for r in rows:
                    se = str(r.get("se", ""))
                    knd_raw = str(r.get("stock_knd") or "").strip()
                    knd = "common" if knd_raw in ("", "-", "nan", "None") else _stock_class(knd_raw)
                    v = _num(r.get(col))
                    if "주당 현금배당금" in se and re.search(r"결산|중간|분기", knd_raw):
                        if v is not None and v > 0:
                            period_sum += v
                        continue
                    if "주당 현금배당금" in se:
                        if knd == "common":
                            if v is not None: out["dps_common"] = v
                            elif str(r.get(col) or "").strip() in ("-", "－"): total_dash0 = True
                    elif "주당순이익" in se:
                        if v is not None: out["eps_reported"] = v
                    elif "현금배당금총액" in se:
                        if v is not None: out["div_total_reported"] = v * 1_000_000
                    elif "당기순이익" in se:
                        if cfs_ni and "별도" in se: continue
                        if v is not None: out["ni_reported"] = v * 1_000_000
                if out.get("dps_common") is None:
                    if period_sum > 0: out["dps_common"] = period_sum
                    elif total_dash0: out["dps_common"] = 0.0
        return True

    if not _fetch_years(probe_years):
        return None
    # 두 탐침 구간(최근 3년 + 6~8년 전) 중 배당을 단 1년이라도 지급한 경우에만 중간·초기 연도 추가 조회
    any_paid = any((v.get("dps_common") or 0) > 0 for v in by_year.values())
    if any_paid and not _fetch_years(rest_years):
        return None
    return by_year

def qualified_in_any_window(by_year: dict[int, dict], start_fy: int, end_fy: int, min_paid: int) -> bool:
    """미래 누출 차단: 2018~2025 어느 한 해라도 직전 5년 중 4년 이상 배당을 지급했으면 2차 수집 대상."""
    for fy in range(start_fy, end_fy + 1):
        paid = sum(1 for y in range(fy - 4, fy + 1) if (by_year.get(y) or {}).get("dps_common") and by_year[y]["dps_common"] > 0)
        if paid >= min_paid:
            return True
    return False

CKPT1_NAME = f"ckpt_pit_stage1_FY{BUILDER_CONFIG['LATEST_FY']}.json.gz"
stage1_map = load_ckpt(CKPT1_NAME)
todo1 = [r.to_dict() for _, r in UNI_DF.iterrows() if r["ticker"] not in stage1_map]

print(f"[2단계] 1차 배당 이력 스캔: 완료 {len(stage1_map):,}/{len(UNI_DF):,} -> 남은 {len(todo1):,}종목")
t0 = time.time()
if todo1:
    try:
        with ThreadPoolExecutor(max_workers=BUILDER_CONFIG["WORKERS"]) as ex:
            futs = {ex.submit(parse_alot_12y, r["corp_code"], BUILDER_CONFIG["LATEST_FY"]): r for r in todo1}
            done_new = 0
            for fut in as_completed(futs):
                r = futs[fut]
                by_y = fut.result()
                if by_y is None:
                    continue   # 한도 도달로 미완료 -> 다음 실행 때 이어받기
                qual = qualified_in_any_window(by_y, BUILDER_CONFIG["PIT_START_FY"], BUILDER_CONFIG["PIT_END_FY"], BUILDER_CONFIG["HIST_MIN_PAID_IN_5Y"])
                stage1_map[r["ticker"]] = {
                    "ticker": r["ticker"], "name": r["name"], "market": r["market"],
                    "path": r["path"], "corp_code": r["corp_code"],
                    "qual": qual, "div": {str(k): v for k, v in by_y.items()},
                }
                done_new += 1
                if done_new % 50 == 0:
                    save_ckpt(CKPT1_NAME, stage1_map)
                    n_q = sum(1 for x in stage1_map.values() if x.get("qual"))
                    print(f"  1차 진행 {len(stage1_map):,}/{len(UNI_DF):,} | 과거배당 적격 {n_q}종목 | DART {DART_STAT['calls']:,}회 ({time.time()-t0:.0f}초, Drive 저장됨)")
    finally:
        save_ckpt(CKPT1_NAME, stage1_map)

QUAL_LIST = [v for v in stage1_map.values() if v.get("qual")]
print(f"[2단계 요약] 완료 {len(stage1_map):,}/{len(UNI_DF):,}종목 | 과거 12년 배당 적격(2차 대상): {len(QUAL_LIST):,}종목 | 금일 DART 호출 {DART_STAT['calls']:,}회")
if DART_STAT["limit_hit"]:
    print("  -> [안내] 오늘 DART 일일 한도(20,000건)에 도달해 Drive에 안전 저장했습니다. 내일 다시 '런타임 -> 모두 실행' 누르면 이어서 진행됩니다!")


In [ ]:
# ==============================================================================
# 5. 3단계: 과거 배당 적격 종목 12년 재무·주식수·자사주 정밀 수집 (15종목마다 Drive 저장)
# ==============================================================================
def extract_fs_year(rows: list[dict], col: str) -> dict:
    def find(ids, nm_pat, sj_pref=None):
        cand = [r for r in rows if r.get("sj_div") in sj_pref] if (sj_pref and any(r.get("sj_div") in sj_pref for r in rows)) else rows
        for aid in ids:
            for r in cand:
                if re.sub(r"^(ifrs-full|ifrs|dart)_", "", str(r.get("account_id") or "")) == aid:
                    v = _num(r.get(col))
                    if v is not None: return v
        if nm_pat:
            rx = re.compile(nm_pat)
            for r in cand:
                if rx.match(str(r.get("account_nm") or "").strip()):
                    v = _num(r.get(col))
                    if v is not None: return v
        return None

    op = find(["OperatingIncomeLoss"], r"^영업(이익|손실|순손익|이익\(손실\))$", ("IS", "CIS"))
    ni_p = find(["ProfitLossAttributableToOwnersOfParent"], r"^지배(기업|주주)(의)?\s*(소유주|지분)?(?!.*(포괄|주당|중단|계속)).*(당기)?순(이익|손익|손실)", ("IS", "CIS"))
    ni = find(["ProfitLoss"], r"^당기순이익(\(손실\))?$", ("IS", "CIS"))
    cfo = find(["CashFlowsFromUsedInOperatingActivities"], r"^영업활동.*현금흐름", ("CF",))
    capex_ppe = find(["PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "PurchaseOfPropertyPlantAndEquipment"], r"유형자산의?\s*(취득|매입|증가)", ("CF",))
    capex_int = find(["PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities", "PurchaseOfIntangibleAssets"], r"무형자산의?\s*(취득|매입|증가)", ("CF",))
    div_paid = find(["DividendsPaidClassifiedAsFinancingActivities", "DividendsPaid"], r"(현금)?배당금의?\s*지급", ("CF",))
    ts_purch = find(["PurchaseOfTreasuryShares", "AcquisitionOfTreasuryShares"], r"자기주식의?\s*취득", ("CF",))
    eq_p = find(["EquityAttributableToOwnersOfParent"], r"^지배(기업|주주)(의)?\s*(소유주|지분)?.*자본", ("BS",))
    eq = find(["Equity"], r"^자본총계$", ("BS",))
    cash = find(["CashAndCashEquivalents"], r"^현금및현금성자산$", ("BS",))
    st_fin = find(["ShortTermFinancialInstruments", "OtherCurrentFinancialAssets"], r"^단기금융상품$", ("BS",))
    st_debt = find(["ShortTermBorrowings"], r"^단기차입금$", ("BS",))
    lt_debt = find(["LongTermBorrowings"], r"^장기차입금$", ("BS",))
    bonds = find(["BondsIssued", "Debentures"], r"^사채$", ("BS",))
    curr_lt = find(["CurrentPortionOfLongTermBorrowings"], r"^유동성장기(부채|차입금)$", ("BS",))
    int_exp = find(["InterestExpense", "FinanceCosts"], r"^이자비용$", ("IS", "CIS"))
    da_d = find(["DepreciationExpense", "Depreciation"], r"^감가상각비$", ("CF", "IS", "CIS"))
    da_a = find(["AmortisationExpense", "AmortizationExpense"], r"^무형자산상각비$", ("CF", "IS", "CIS"))

    capex = (abs(capex_ppe) if capex_ppe is not None else 0.0) + (abs(capex_int) if capex_int is not None else 0.0) if (capex_ppe is not None or capex_int is not None) else None
    fcf = (cfo - capex) if (cfo is not None and capex is not None) else None
    eq_use = eq_p if eq_p is not None else eq
    ni_use = ni_p if ni_p is not None else ni
    cash_t = (cash or 0.0) + (st_fin or 0.0) if (cash is not None or st_fin is not None) else None
    debt_vals = [x for x in (st_debt, lt_debt, bonds, curr_lt) if x is not None]
    debt_t = sum(debt_vals) if debt_vals else 0.0
    net_debt = (debt_t - cash_t) if cash_t is not None else None
    da = (abs(da_d) if da_d is not None else 0.0) + (abs(da_a) if da_a is not None else 0.0) if (da_d is not None or da_a is not None) else None
    ebitda = (op + da) if (op is not None and da is not None) else op
    icr = (op / abs(int_exp)) if (op is not None and int_exp is not None and abs(int_exp) > 0) else None
    nde = (net_debt / ebitda) if (net_debt is not None and ebitda is not None and ebitda > 0) else None
    roe = (ni_use / eq_use) if (ni_use is not None and eq_use is not None and eq_use > 0) else None
    ic = (eq_use + max(0.0, net_debt)) if (eq_use is not None and net_debt is not None) else eq_use
    roic = (op * 0.78 / ic) if (op is not None and ic is not None and ic > 0) else None

    return {
        "net_income_parent": ni_use, "fcf": fcf,
        "dividends_paid": abs(div_paid) if div_paid is not None else None,
        "treasury_purchase": abs(ts_purch) if ts_purch is not None else None,
        "equity_parent": eq_use, "roe": roe, "roic": roic,
        "net_debt": net_debt, "net_debt_to_ebitda": nde, "interest_coverage": icr,
    }

def collect_stage2_full_one(item: dict, last_fy: int) -> dict | None:
    corp = item["corp_code"]
    # 1) 재무제표 (4회 호출 = 12년)
    fs_by_year = {}
    for cy in [y for y in (last_fy, last_fy - 3, last_fy - 6, last_fy - 9) if y >= 2015]:
        rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "CFS"})
        if rows is None: return None
        if not rows:
            rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "OFS"})
            if rows is None: return None
        if not rows: continue
        for col, lag in (("thstrm_amount", 0), ("frmtrm_amount", 1), ("bfefrmtrm_amount", 2)):
            y = cy - lag
            if y not in fs_by_year:
                ext = extract_fs_year(rows, col)
                if any(v is not None for v in ext.values()):
                    fs_by_year[y] = ext

    # 2) 연도별 발행주식수 (PIT 시총 & 희석 계산용: 2015, 2018~last_fy)
    sh_by_year = {}
    sh_years = [2015] + list(range(BUILDER_CONFIG["PIT_START_FY"], last_fy + 1))
    for y in sorted(set(sh_years)):
        rows = dart_get("stockTotqySttus", corp, y)
        if rows is None: return None
        if not rows: continue
        rcp = max(str(r.get("rcept_no") or "") for r in rows)
        oc, op_sh = 0.0, 0.0
        for r in rows:
            if str(r.get("rcept_no") or "") != rcp: continue
            k = _stock_class(r.get("se", ""))
            v = _num(r.get("distb_stock_co"))
            if k == "common" and v: oc += v
            elif k == "pref" and v: op_sh += v
        if oc > 0:
            sh_by_year[y] = {"common": oc, "pref": op_sh}

    # 3) 자기주식 취득·소각 조건부 조회 (현금흐름표 자사주취득 > 0 또는 주식수 변동이 있는 연도만 호출 -> ~6,500회 절감!)
    ts_by_year = {}
    for y in range(2015, last_fy + 1):
        tp_y = (fs_by_year.get(y) or {}).get("treasury_purchase") or 0.0
        sh_now = (sh_by_year.get(y) or {}).get("common")
        sh_prev = (sh_by_year.get(y - 1) or {}).get("common")
        sh_changed = bool(sh_now and sh_prev and abs(sh_now - sh_prev) / max(1.0, sh_prev) > 0.002)
        if tp_y <= 0 and not sh_changed and y < last_fy:
            ts_by_year[y] = {"acq": 0.0, "canc": 0.0}
            continue
        rows = dart_get("tesstkAcqsDspsSttus", corp, y)
        if rows is None: return None
        if not rows:
            ts_by_year[y] = {"acq": 0.0, "canc": 0.0}
            continue
        acq, canc = 0.0, 0.0
        for r in rows:
            if _stock_class(r.get("stock_knd", "")) == "common" and str(r.get("acqs_mth1")) == "총계":
                acq += _num_dash0(r.get("change_qy_acqs")) or 0.0
                canc += _num_dash0(r.get("change_qy_incnr")) or 0.0
        ts_by_year[y] = {"acq": acq, "canc": canc}

    return {
        "fs": {str(k): v for k, v in fs_by_year.items()},
        "sh": {str(k): v for k, v in sh_by_year.items()},
        "ts": {str(k): v for k, v in ts_by_year.items()},
    }

CKPT2_NAME = f"ckpt_pit_stage2_FY{BUILDER_CONFIG['LATEST_FY']}.json.gz"
stage2_map = load_ckpt(CKPT2_NAME)

if len(stage1_map) < len(UNI_DF) and DART_STAT["limit_hit"]:
    print("[대기] 1차 스캔이 아직 남아 있어 2차 수집은 다음 실행에서 이어서 진행합니다.")
else:
    todo2 = [it for it in QUAL_LIST if it["ticker"] not in stage2_map]
    print(f"[3단계] 2차 정밀 재무·주식수·자사주 수집: 완료 {len(stage2_map):,}/{len(QUAL_LIST):,} -> 남은 {len(todo2):,}종목")
    t0 = time.time()
    if todo2 and not DART_STAT["limit_hit"]:
        try:
            with ThreadPoolExecutor(max_workers=BUILDER_CONFIG["WORKERS"]) as ex:
                futs = {ex.submit(collect_stage2_full_one, it, BUILDER_CONFIG["LATEST_FY"]): it for it in todo2}
                done_new = 0
                for fut in as_completed(futs):
                    it = futs[fut]
                    res = fut.result()
                    if res is None:
                        continue   # 한도 도달
                    stage2_map[it["ticker"]] = res
                    done_new += 1
                    if done_new % 15 == 0:
                        save_ckpt(CKPT2_NAME, stage2_map)
                        pct = len(stage2_map) / max(1, len(QUAL_LIST)) * 100
                        print(f"  2차 진행 {len(stage2_map):,}/{len(QUAL_LIST):,} ({pct:.1f}%) | DART {DART_STAT['calls']:,}회 ({time.time()-t0:.0f}초, Drive 저장됨)")
        finally:
            save_ckpt(CKPT2_NAME, stage2_map)

    pct2 = len(stage2_map) / max(1, len(QUAL_LIST)) * 100
    print(f"[3단계 요약] 완료 {len(stage2_map):,}/{len(QUAL_LIST):,}종목 ({pct2:.1f}%) | 금일 총 DART 호출 {DART_STAT['calls']:,}회")
    if DART_STAT["limit_hit"] and len(stage2_map) < len(QUAL_LIST):
        print("  -> [안전 일시정지] 오늘 OpenDART 20,000건 한도에 도달해 Drive에 저장 완료했습니다.")
        print("     내일 다시 '런타임 -> 모두 실행'을 누르면 남은 종목만 이어받아 자동으로 완성됩니다!")


In [ ]:
# ==============================================================================
# 6. 4단계: 액면분할 보정 DPS 산출 & 최종 풀 유니버스 PIT DB(.jsonl.gz) Drive 출력
# ==============================================================================
def compute_split_factors(sh_dict: dict[int, dict], latest_fy: int) -> dict[int, float]:
    """연도별 보통주 유통주식수 급증(>=1.8배: 액면분할·무상증자)을 자동 감지해 최신 연도 기준 분할비율 계산."""
    ys = sorted(y for y, d in sh_dict.items() if (d.get("common") or 0) > 0)
    if not ys:
        return {}
    # 뒤에서 앞으로 누적 분할배수 계산 (연간 1.8배 이상 급증 시 분할/무상증자로 간주)
    cum = 1.0
    fac = {ys[-1]: 1.0}
    for i in range(len(ys) - 2, -1, -1):
        y_prev, y_next = ys[i], ys[i + 1]
        c_prev, c_next = sh_dict[y_prev]["common"], sh_dict[y_next]["common"]
        ratio = c_next / c_prev if c_prev > 0 else 1.0
        if ratio >= 1.8:
            # 정수 분할(2, 5, 10, 50 등) 근사 시 깔끔한 배수 사용
            nearest = round(ratio)
            step = float(nearest) if abs(ratio - nearest) / ratio < 0.08 else ratio
            cum *= step
        fac[y_prev] = cum
    # 중간 빈 연도 채우기
    out = {}
    for y in range(2013, latest_fy + 1):
        # y 이상 중 가장 가까운 기준연도의 배수 사용
        cands = [k for k in ys if k >= y]
        out[y] = fac[cands[0]] if cands else 1.0
    return out

def assemble_pit_record(s1: dict, s2: dict) -> dict:
    div_y = {int(k): v for k, v in (s1.get("div") or {}).items()}
    fs_y = {int(k): v for k, v in (s2.get("fs") or {}).items()}
    sh_y = {int(k): v for k, v in (s2.get("sh") or {}).items()}
    ts_y = {int(k): v for k, v in (s2.get("ts") or {}).items()}
    last_fy = BUILDER_CONFIG["LATEST_FY"]

    split_fac = compute_split_factors(sh_y, last_fy)
    act_dps = {}
    for y, dv in div_y.items():
        dp = dv.get("dps_common")
        if dp is not None:
            act_dps[str(y)] = round(float(dp) / split_fac.get(y, 1.0), 4)

    # 연도별 통합 스냅샷 (Point-in-Time: fy 시점에는 <= fy 데이터만 포함!)
    all_y_rec = {}
    for y in range(2013, last_fy + 1):
        dv = div_y.get(y) or {}
        fs = fs_y.get(y) or {}
        sh = sh_y.get(y) or {}
        ts = ts_y.get(y) or {}
        rec_y = {
            "dps_common": dv.get("dps_common"),
            "eps_reported": dv.get("eps_reported"),
            "net_income_parent": fs.get("net_income_parent") if fs.get("net_income_parent") is not None else dv.get("ni_reported"),
            "fcf": fs.get("fcf"),
            "dividends_paid": fs.get("dividends_paid") if fs.get("dividends_paid") is not None else dv.get("div_total_reported"),
            "treasury_purchase": fs.get("treasury_purchase"),
            "equity_parent": fs.get("equity_parent"),
            "roe": fs.get("roe"),
            "roic": fs.get("roic"),
            "net_debt": fs.get("net_debt"),
            "net_debt_to_ebitda": fs.get("net_debt_to_ebitda"),
            "interest_coverage": fs.get("interest_coverage"),
            "outstanding_common": sh.get("common"),
            "outstanding_pref": sh.get("pref"),
            "ts_acquired_common": ts.get("acq"),
            "ts_cancelled_common": ts.get("canc"),
            "split_factor_to_latest": split_fac.get(y, 1.0),
        }
        if any(v is not None for k, v in rec_y.items() if k != "split_factor_to_latest"):
            all_y_rec[str(y)] = rec_y

    pit_frames = {}
    for fy in range(BUILDER_CONFIG["PIT_START_FY"], BUILDER_CONFIG["PIT_END_FY"] + 1):
        pit_frames[str(fy)] = {yk: yv for yk, yv in all_y_rec.items() if int(yk) <= fy}

    return {
        "ticker": s1["ticker"],
        "name": s1["name"],
        "market": s1["market"],
        "path": s1["path"],
        "act_dps": act_dps,
        "pit": pit_frames,
    }

ready_tks = [it["ticker"] for it in QUAL_LIST if it["ticker"] in stage2_map]
if not ready_tks:
    print("[대기] 아직 2차 수집 완료 종목이 없습니다. 내일 다시 실행해 이어받아 주세요.")
else:
    lines = []
    for tk in sorted(ready_tks):
        rec = assemble_pit_record(stage1_map[tk], stage2_map[tk])
        lines.append(json.dumps(rec, ensure_ascii=False, separators=(",", ":")))
    out_bytes = gzip.compress(("\n".join(lines) + "\n").encode("utf-8"), compresslevel=6)
    out_name = BUILDER_CONFIG["OUTPUT_FILENAME"]
    (CACHE_DIR / out_name).write_bytes(out_bytes)
    if DRIVE_ROOT is not None:
        tmp = DRIVE_ROOT / (out_name + ".tmp")
        dst = DRIVE_ROOT / out_name
        tmp.write_bytes(out_bytes)
        tmp.replace(dst)
        print(f"[DB 저장 완료] {dst} ({len(lines):,}개사, {len(out_bytes)/1024:,.1f} KB)")
    else:
        print(f"[로컬 DB 저장 완료] {CACHE_DIR / out_name} ({len(lines):,}개사, {len(out_bytes)/1024:,.1f} KB)")

    if len(ready_tks) == len(QUAL_LIST) and len(stage1_map) == len(UNI_DF):
        print("  -> [100% 완성!] 이제 colab_alpha_lab.ipynb를 실행하면 자동으로 이 풀 유니버스 DB를 읽어 분석합니다!")
    else:
        print(f"  -> [부분 저장됨] 현재 {len(ready_tks):,}/{len(QUAL_LIST):,}개사 반영. 내일 다시 실행하면 나머지 종목까지 자동 합쳐집니다!")
